# Multilayer Perceptron (MLP) Example (Wine Quality Dataset)

Here it is demonstrated how to use the `MLPClassifier` module from the CMOR-438 library to perform binary classification with a neural network.
In this example, the Wine Quality dataset is used to train, test, and evaluate the model.

**Goal: Predict whether a wine is High Quality (score ≥ 7) using a two-hidden-layer neural network.**

The MLP architecture used:
- **Input layer:** 11 features
- **Hidden layer 1:** 64 neurons (ReLU)
- **Hidden layer 2:** 32 neurons (ReLU)
- **Output layer:** 1 neuron (Sigmoid → probability)

## 1. Setup and Data Loading

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, '../_shared')
from multilayer_perceptron import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv('../../../data/WineQT.csv').drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")

## 2. Preprocessing

Create a binary target (High Quality vs rest), standardise features, and split 80/20.

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_bin = (wine['quality'].values >= 7).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")
print(f"Class balance — Low/Mid: {(y_bin==0).sum()}  High: {(y_bin==1).sum()}")

## 3. Train

Train a two-hidden-layer MLP using mini-batch backpropagation.
L2 regularisation is applied to prevent overfitting.

In [ ]:
mlp = MLPClassifier(hidden_layers=(64, 32), learning_rate=0.05, n_iterations=400, l2=1e-4)
mlp.fit(X_tr, y_tr)
print(f'MLP Accuracy: {mlp.accuracy(X_te, y_te):.4f}')

## 4. Results and Visualisation

Two plots are produced:
- **Training loss curve** — Binary Cross-Entropy over epochs; a smooth downward curve indicates stable learning
- **Predicted probability distribution** — P(High Quality) for each true class; good separation between the two distributions indicates a confident classifier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(mlp.loss_history_, color='teal', lw=1.5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('MLP Training Loss', fontweight='bold')

probs = mlp.predict_proba(X_te)
for label, color, name in zip([0,1],['steelblue','darkorange'],['Low/Mid','High Quality']):
    axes[1].hist(probs[y_te==label], bins=25, alpha=0.6, color=color, label=name)
axes[1].axvline(0.5, color='red', linestyle='--', lw=1.5, label='Threshold')
axes[1].set_xlabel('P(High Quality)'); axes[1].set_ylabel('Count')
axes[1].set_title('MLP Predicted Probability Distribution', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()